Worksheet to assess the status of the simulations run by  
* StefanProblem_Run.ipynb  
* SuckingProblem_Run.ipynb  
* Filmboiling_Run.ipynb  
* ScrivenProblem_Run.ipynb  

In [1]:
#r "./binaries/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

Valid project names:  

* StefanProblem  
* SuckingProblem  
* Filmboiling  
    * Filmboiling_v1 (run with alogim and saye, repeat run, I think it was a spurious fail)
    * Filmboiling_v2 (Saye and 1 Core, 1 Thread)
* ScrivenProblem  
    * ScrivenProblem_v1 (added reinitialization)
    * ScrivenProblem_v2 (reduced amr on narrowband bandwidth from 1 to 0)
    * ScrivenProblem_v3 (changed to pardiso)

In [ ]:
string[] projects = new string[] {
    "StefanProblem", 
    "SuckingProblem",
    "Filmboiling",
    "ScrivenProblem"
};

In [3]:
Dictionary<string,Dictionary<string,ISessionInfo>> FailedSession = new Dictionary<string,Dictionary<string,ISessionInfo>>();

In [4]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();
foreach(string proj in projects){
    FailedSession[proj] = new Dictionary<string,ISessionInfo>();
    Console.WriteLine("====================================================");
    BoSSSshell.WorkflowMgm.Init(proj);
    var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
    var fsessions = sessions.Where(s => !s.SuccessfulTermination);

    Console.WriteLine("Found {0,3} sessions for {1,-14}", sessions.Count(), proj);
    Console.WriteLine("{0,3} sessions succesful", sessions.Where(s => s.SuccessfulTermination).Count());
    Console.WriteLine("{0,3} sessions failed", sessions.Where(s => !s.SuccessfulTermination).Count());

    if(fsessions.Count() > 0){
        Console.WriteLine("Failed for : ");
        foreach(var s in fsessions){
            FailedSession[proj][s.ID.ToString()] = s;
            Console.WriteLine("{0} : {1}", s.ID, s.Name);
            Console.WriteLine("at", s.ID, s.Name);
            Console.WriteLine("{0}", Path.Combine(s.Database.Path, "sessions", s.ID.ToString()));
        }
    }
}

In [ ]:
FailedSession["ScrivenProblem_v1"].Values.ForEach( s => s.Export().To(Path.GetFullPath(Path.Combine(".","Plots",s.ProjectName,s.Name))).WithSupersampling(2).Do());

In [5]:
//FailedSession["Filmboiling"].Values.ForEach( s => s.Export().To(Path.GetFullPath(Path.Combine(".","Plots","Filmboiling",s.Name))).WithSupersampling(2).Do());

In [6]:
FailedSession["Filmboiling"].Values.First().NumberOfThreadsPerRank()

4

In [7]:
BoSSSshell.WorkflowMgm.Init("Filmboiling");
var C = FailedSession["Filmboiling"].Values.First().CreateRestartControl();
C.saveperiod=1;
C.ProjectName = "Filmboiling";
C.GridGuid = FailedSession["Filmboiling"].Values.First().Timesteps.Last().GridID;


Project name is set to 'Filmboiling'.
Default Execution queue is chosen for the database.


In [8]:
var J = C.CreateJob();
J.Activate();

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Filmboiling_H2_AMR4_P3_Convergence_Restart ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\rieckmann\cluster\binaries\Filmboiling-XNSFE_Solver2025May28_111512.302978
copied 71 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


In [ ]:
BoSSSshell.WorkflowMgm.Sessions

In [15]:
BoSSSshell.WorkflowMgm.Sessions.Take(1).ForEach( s => s.Timesteps.Skip(20).Export().To(Path.GetFullPath(Path.Combine(".","Plots","Filmboiling",s.Name))).WithSupersampling(2).Do());

Starting export process... Data will be written to the directory: d:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-Merging\public\examples\XNSFE_Solver\EvaporationValidation\Plots\Filmboiling\Filmboiling_H2_AMR4_P3_Convergence_Restart


In [17]:
FailedSession["Filmboiling"].Values.First().GetApproximateRunTime().Display();
FailedSession["Filmboiling"].Values.First().GetAverageCPUTimePerTimestep().Display();
FailedSession["Filmboiling"].Values.First().GetAverageComputingTimePerTimestep().Display();

5.23:13:14.2160000

51.81851417085427

51.81851417085427